# Week 28, Day 4: Methods and self

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w028d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Counter:
    def __init__(self):
        self.n = 0

    def add(self, k=1):
        self.n += k

c = Counter()
c.add()
Counter.add(c, 5)
press = c.add
press()
press()
print(c.n)

**Predict before you run:** what will this print?



In [ ]:
class Shelf:
    def __init__(self):
        self.books = []

    def add(self, title):
        self.books.append(title.strip().title())

s = Shelf()
for t in ["  dune", "emma "]:
    s.add(t)
list(map(s.add, ["persuasion", "middlemarch"]))
print(s.books)
words = "the cat sat".split()
print(list(map(str.upper, words)))

In [ ]:
class Account:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance

    def can_pay(self, amount):
        return 0 < amount <= self.balance

    def withdraw(self, amount):
        if not self.can_pay(amount):
            raise ValueError(f"{self.owner} can't pay {amount}")
        self.balance -= amount

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("deposit must be positive")
        self.balance += amount

    def transfer(self, other, amount):
        self.withdraw(amount)
        other.deposit(amount)

ada, bo = Account("Ada", 100), Account("Bo", 20)
ada.transfer(bo, 30)
print(ada.balance, bo.balance)
try:
    bo.transfer(ada, 500)
except ValueError as e:
    print(e)
print(ada.balance, bo.balance)

**Predict before you run:** what will this print?



In [ ]:
class Queue:
    def __init__(self):
        self.items = []

    def join(self, name):
        self.items.append(name)

    def front(self):
        return self.items[0] if self.items else None

    def serve(self):
        return self.items.pop(0)

q = Queue()
q.join("Ada"); q.join("Bo")
print(q.front(), q.front(), q.serve(), q.front(), len(q.items))

In [ ]:
class Basket:
    def __init__(self):
        self._items = {}

    def _check(self, qty):
        if type(qty) is not int or qty <= 0:
            raise ValueError(f"quantity must be a positive whole number, not {qty!r}")

    def add(self, item, qty=1):
        self._check(qty)
        self._items[item] = self._items.get(item, 0) + qty

    def remove(self, item, qty=1):
        self._check(qty)
        have = self._items.get(item, 0)
        if qty > have:
            raise ValueError(f"only {have} {item} in the basket")
        if qty == have:
            del self._items[item]
        else:
            self._items[item] = have - qty

    def count(self, item):
        return self._items.get(item, 0)

    def lines(self):
        return sorted(self._items.items())

b = Basket()
b.add("apple", 3)
b.add("pear")
b.remove("apple", 3)
print(b.lines(), b.count("apple"))
for bad in [("pear", 0), ("pear", 5)]:
    try:
        b.remove(*bad)
    except ValueError as e:
        print(e)

> **Key idea.** `obj.method(x)` is `Class.method(obj, x)`: `self` is the instance. `obj.method` without brackets is a bound method. Methods call each other through `self.`, and can work with other instances. Queries answer and change nothing; commands change and return `None`. Keep invariants true by checking before changing, and mark internal helpers and attributes with a leading underscore.

### Exercise: A stopwatch

Write `Stopwatch` with methods taking the current time in seconds (so they're easy to test): `start(now)`, `stop(now)` and `elapsed(now)`. It can be started and stopped many times, adding up the running time. `elapsed(now)` includes the current run if it's running. Starting while running, or stopping while stopped, raises `RuntimeError`.

In [ ]:
class Stopwatch:
    def __init__(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_runs():\n    \"adds up runs\"\n    S = ns[\"Stopwatch\"]\n    w = S()\n    assert w.elapsed(100) == 0\n    w.start(10)\n    assert w.elapsed(15) == 5\n    w.stop(20)\n    assert w.elapsed(1000) == 10\n    w.start(50)\n    w.stop(55.5)\n    assert w.elapsed(60) == 15.5\ndef test_errors():\n    \"wrong order\"\n    S = ns[\"Stopwatch\"]\n    w = S()\n    try:\n        w.stop(1)\n        assert False, \"stopping a stopped stopwatch should raise RuntimeError\"\n    except RuntimeError:\n        pass\n    w.start(1)\n    try:\n        w.start(2)\n        assert False, \"starting a running stopwatch should raise RuntimeError\"\n    except RuntimeError:\n        pass")

### Exercise: Transfers between objects

Write `Wallet(owner, pence=0)` with `pay(other, amount)`: move `amount` pence to another wallet, raising `ValueError` (and changing nothing) if the amount isn't positive, if it's more than this wallet holds, or if `other` is this same wallet. Every wallet keeps a `log` list of strings: the payer gets `"paid Bo 250"` and the receiver `"received 250 from Ada"`.

In [ ]:
class Wallet:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pay():\n    \"paying\"\n    W = ns[\"Wallet\"]\n    a, b = W(\"Ada\", 1000), W(\"Bo\")\n    a.pay(b, 250)\n    b.pay(a, 50)\n    assert (a.pence, b.pence) == (800, 200)\n    assert a.log == [\"paid Bo 250\", \"received 50 from Bo\"] and b.log == [\"received 250 from Ada\", \"paid Ada 50\"]\ndef test_refused():\n    \"refused payments change nothing\"\n    W = ns[\"Wallet\"]\n    a, b = W(\"Ada\", 100), W(\"Bo\", 5)\n    for other, amount in [(b, 0), (b, -5), (b, 101), (a, 10)]:\n        try:\n            a.pay(other, amount)\n            assert False, f\"paying {amount} should raise ValueError\"\n        except ValueError:\n            pass\n    assert (a.pence, b.pence, a.log, b.log) == (100, 5, [], [])")

### Exercise: Methods that use methods

Complete `Grades`: `add(subject, mark)` stores a mark (0 to 100, or `ValueError`; adding a subject again replaces its mark); `average()` (rounded to 1 place, `None` with no marks); `grade()` returns the letter for the average using `letter(mark)`: A from 70, B from 60, C from 50, otherwise F, or `None` with no marks; and `report()` returns lines like `maths 72 A`, subjects A to Z, then `average 64.5 B`. Use `self.` to call `letter` and `average` rather than repeating their code.

In [ ]:
class Grades:
    def __init__(self):
        self._marks = {}

    def letter(self, mark):
        ___

    def add(self, subject, mark):
        ___

    def average(self):
        ___

    def grade(self):
        ___

    def report(self):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_report():\n    \"report\"\n    g = ns[\"Grades\"]()\n    g.add(\"science\", 57)\n    g.add(\"maths\", 72)\n    g.add(\"maths\", 72)\n    assert g.average() == 64.5 and g.grade() == \"B\"\n    assert g.report() == [\"maths 72 A\", \"science 57 C\", \"average 64.5 B\"]\ndef test_empty_and_errors():\n    \"empty, and bad marks\"\n    g = ns[\"Grades\"]()\n    assert g.average() is None and g.grade() is None and g.report() == []\n    for bad in (-1, 101):\n        try:\n            g.add(\"art\", bad)\n            assert False, f\"a mark of {bad} should raise ValueError\"\n        except ValueError:\n            pass")

### Exercise: Bound methods

Using the `Shelf` class, add each title in `incoming` to `shelf` with `map` and the bound method (no loop), then print how many books there are and the sorted titles, using `sorted` with the **unbound** method `str.lower` as the key.

In [ ]:
class Shelf:
    def __init__(self):
        self.books = []

    def add(self, title):
        self.books.append(title.strip())

shelf = Shelf()
incoming = ["emma", " Dune", "persuasion ", "Anna Karenina"]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"output\"\n    expect_output(\"4 ['Anna Karenina', 'Dune', 'emma', 'persuasion']\")\n    assert \"map(shelf.add\" in source.replace(\" \", \"\") or \"map(shelf.add\" in source\n    assert \"key=str.lower\" in source.replace(\" \", \"\")")

### Bug hunt: Not defined?

`checkout` crashes with `NameError: name 'subtotal' is not defined`. Fix it.

In [ ]:
class Till:
    def __init__(self):
        self.prices = []

    def scan(self, price):
        self.prices.append(price)

    def subtotal(self):
        return sum(self.prices)

    def checkout(self, discount=0):
        return subtotal() - discount

t = Till()
t.scan(250)
t.scan(199)
print(t.checkout(50))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"total\"\n    expect_output(\"399\")")

### Bug hunt: A question that changes the answer

`latest()` is meant to be a query: look at the most recent message without changing anything. But calling it twice gives different answers. Fix it so it only looks.

In [ ]:
class Inbox:
    def __init__(self):
        self.messages = []

    def receive(self, text):
        self.messages.append(text)

    def latest(self):
        return self.messages.pop() if self.messages else None

box = Inbox()
box.receive("hello")
box.receive("are you there?")
print(box.latest(), "|", box.latest(), "|", len(box.messages))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only looks\"\n    expect_output(\"are you there? | are you there? | 2\")")

### Puzzle: Reuse a check

Arrange the lines into a class where `withdraw` uses `can_pay`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class Light:
    def __init__(self):
        self.on = False
        self.switches = 0

    def toggle(self):
        self.on = not self.on
        self.switches += 1

    def is_on(self):
        return self.on

a, b = Light(), Light()
flip = a.toggle
flip(); flip(); flip()
Light.toggle(b)

p = ___   # a.is_on()
q = ___   # a.switches
r = ___   # b.is_on()
s = ___   # what a.toggle() returns (the Python value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r, s) == (True, 3, True, None)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a lift

Write `Lift(lowest, highest)` for a lift starting at `lowest`, doors closed, going nowhere. `call(floor)` adds a floor to visit (`ValueError` if outside the range; calling a floor already waiting, or the current floor while stopped, changes nothing). `step()` moves one floor towards the next stop and returns the new floor; arriving at a waiting floor removes it from the waiting set. When choosing where to go, the lift keeps going in its current direction while there are waiting floors that way, and only then turns round (like real lifts). With nothing waiting, `step()` returns the current floor and the lift stops. Also write `waiting()` returning the waiting floors, sorted.

In [ ]:
class Lift:
    def __init__(self, lowest, highest):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def run(lift, n):\n    return [lift.step() for _ in range(n)]\ndef test_simple():\n    \"up and back\"\n    L = ns[\"Lift\"]\n    lift = L(0, 10)\n    lift.call(3)\n    assert run(lift, 4) == [1, 2, 3, 3]\n    lift.call(1)\n    assert run(lift, 2) == [2, 1] and lift.waiting() == []\ndef test_direction():\n    \"keeps going while there are floors that way\"\n    L = ns[\"Lift\"]\n    lift = L(0, 10)\n    lift.call(5)\n    run(lift, 2)\n    lift.call(1)\n    lift.call(7)\n    assert run(lift, 5) == [3, 4, 5, 6, 7]\n    assert lift.waiting() == [1]\n    assert run(lift, 6) == [6, 5, 4, 3, 2, 1] and lift.waiting() == []\ndef test_errors():\n    \"range and repeats\"\n    L = ns[\"Lift\"]\n    lift = L(-1, 3)\n    for bad in (-2, 4):\n        try:\n            lift.call(bad)\n            assert False, f\"call({bad}) should raise ValueError\"\n        except ValueError:\n            pass\n    lift.call(-1)\n    lift.call(2)\n    lift.call(2)\n    assert lift.waiting() == [2]")

## Warm-up

Three ways to call the same method, all doing the same thing. `c.add()` is short for `Counter.add(c)`, and `c.add` on its own (without brackets) is a **bound method**: the function with `c` already attached, ready to call later.

## What a method call really is

A method is just a function defined inside a class. When you write `c.add(5)`, Python:

1. finds `add` in `c`'s class;
2. calls it with `c` as the first argument: `Counter.add(c, 5)`.

That first parameter is called `self` by universal convention (Python doesn't force the name, but every reader expects it). It's how a method knows **which** instance it's working on: the same `add` code updates `c` or any other counter, depending on which one you called it on.

Bound methods are useful wherever a function is expected:

`str.upper` is the method looked up on the class, so it needs the string passed in; `s.add` is bound to `s` already.

## Methods working together

Inside a method, `self` gives access to the instance's attributes **and** its other methods. Calling `self.other_method()` keeps each method small and each rule written once.

`transfer` reuses `withdraw` and `deposit`, so it gets their checks for free. It also works with **another** instance, `other`, through that object's own methods. And because `withdraw` raises before `deposit` runs, a failed transfer changes nothing.

Forgetting `self.` is the most common slip: `can_pay(amount)` on its own looks for a plain function called `can_pay` and fails with `NameError`.

## Questions and commands

A clear habit for methods, known as **command-query separation**:

- a **query** answers a question and changes nothing: `can_pay`, `area`, `is_empty`, `total`. You can call it as often as you like.
- a **command** changes the object and returns `None`: `deposit`, `add`, `clear`, `sort`.

Python's own types mostly follow it: `list.sort()` changes the list and returns `None`; `sorted()` changes nothing and returns an answer. Mixing the two causes surprises, like a "get" method that quietly removes what it returns, so asking twice gives a different answer.

The main exception is a method that both removes and returns by design, with a name that says so: `list.pop()`, `dict.pop()`. If you write one, name it so the change is obvious.

## Keeping the rules

Most classes have rules that should **always** be true for every instance: a balance never negative, a basket's quantities always positive, a queue never holding the same person twice. These are called **invariants**. Three habits keep them:

1. `__init__` makes them true from the start (yesterday).
2. Every method that changes state checks it keeps them true, and raises **before** changing anything if it wouldn't.
3. Code outside the class changes the object only through its methods, not by assigning attributes directly.

Python doesn't enforce the third. Instead, a naming convention signals intent: a name starting with **one underscore**, like `self._items` or `def _check(self)`, means "internal: not for use outside this class". Readers and linters respect it. Next week covers this properly, with properties that let you control access.

Whatever sequence of method calls you try, every quantity in the basket stays a positive whole number. That guarantee is what makes the class trustworthy.

## Practice

## Quiz

## Daily challenge

## Before you go

Take one of your classes and sort its methods into queries and commands. Does any method do both? Could any code outside the class break its rules by setting an attribute directly? Tomorrow: attributes that belong to the class itself, shared by every instance.